In [ ]:
import arcpy
import numpy as np
import pandas as pd

In [ ]:
fc = r"path\to\your\ArcGISProgeodatabase.gdb\popularity_201801_202001"

In [ ]:
fields_list = [field.name for field in arcpy.ListFields(fc)]

In [ ]:
df = pd.DataFrame.from_records(
    data=arcpy.da.SearchCursor(fc, fields_list),
    columns=fields_list
)

In [ ]:
deer_columns = [
    'Jan18_deer', 'Feb18_deer', 'Mar18_deer', 'Apr18_deer', 'May18_deer',
    'Jun18_deer', 'Jul18_deer', 'Aug18_deer', 'Sep18_deer', 'Oct18_deer',
    'Nov18_deer', 'Dec18_deer', 'Jan19_deer', 'Feb19_deer', 'Mar19_deer',
    'Apr19_deer', 'May19_deer', 'Jun19_deer', 'Jul19_deer', 'Aug19_deer',
    'Sep19_deer', 'Oct19_deer', 'Nov19_deer', 'Dec19_deer'
]

human_columns = [
    'Jan18_hum', 'Feb18_hum', 'Mar18_hum', 'Apr18_hum', 'May18_hum',
    'Jun18_hum', 'Jul18_hum', 'Aug18_hum', 'Sep18_hum', 'Oct18_hum',
    'Nov18_hum', 'Dec18_hum', 'Jan19_hum', 'Feb19_hum', 'Mar19_hum',
    'Apr19_hum', 'May19_hum', 'Jun19_hum', 'Jul19_hum', 'Aug19_hum',
    'Sep19_hum', 'Oct19_hum', 'Nov19_hum', 'Dec19_hum'
]

In [ ]:
flattened_data = []

for _, row in df.iterrows():
    for deer_col, human_col in zip(deer_columns, human_columns):
        # extract the month-year from the column name (formatted as MMMYY)
        month_year = deer_col.split('_')[0]

        # convert popularity values from a string list to an integer list
        popdeer_values = map(int, row[deer_col][1:-1].split(','))
        pophuman_values = map(int, row[human_col][1:-1].split(','))

        # create 24 records (one per hour) for each month, for each grid ID cell
        flattened_data.extend(
            {
                'GRID_ID_p': row['GRID_ID_p'],
                'month_year': month_year,
                'hour': hour,
                'popdeer': popdeer,
                'pophuman': pophuman
            }
            for hour, (popdeer, pophuman) in enumerate(
                zip(popdeer_values, pophuman_values)
            )
        )

flattened_df = pd.DataFrame(flattened_data)

In [ ]:
# calculate the total human popularity for each grid ID
pophuman_sum_per_grid = flattened_df.groupby('GRID_ID_p')['pophuman'].sum()

landsctype_mapping = {}

for grid_id, pophuman_sum in pophuman_sum_per_grid.items():
    # if the grid ID starts with 'Park', categorize it as 2 (parks)
    if grid_id.startswith('Park'):
        landsctype_mapping[grid_id] = 2
    # if the the total human popularity is 0 for this grid,
    # categorize it as 0 (no human commercial activity)
    elif pophuman_sum == 0:
        landsctype_mapping[grid_id] = 0
    # otherwise, categorize it as 1 (areas with human commercial activity)
    else:
        landsctype_mapping[grid_id] = 1

# add landscape type column
flattened_df['landsctype'] = flattened_df['GRID_ID_p'].map(landsctype_mapping)

In [ ]:
# calculate the hourly means of both deer and human popularity
# across each landscape type
# 0: areas with no human commercial activity
# 1: areas with human commercial activity
# 2: parks

grouped_df = (
    flattened_df.groupby(['hour', 'landsctype'])
    .agg({'popdeer': 'mean', 'pophuman': 'mean'})
    .reset_index()
    .rename(columns={'popdeer': 'popdeer_mean', 'pophuman': 'pophuman_mean'})
)

In [ ]:
# create a dictionary for the above-calculated values
# Note: Fig. 3 applied smoothing (3-hour rolling mean) 
# to the data before plotting and y-tick labels were
# rounded to one decimal place

output_dict = {}

for lstype_fig2 in [0, 1, 2]:
    ls_data = grouped_df[grouped_df['landsctype'] == lstype_fig2]
    ls_dict = {
        'popdeer_mean': ls_data['popdeer_mean'].tolist(),
        'pophuman_mean': ls_data['pophuman_mean'].tolist()
    }
    output_dict[lstype_fig2] = ls_dict